# End of week 1 exercise

To demonstrate your familiarity with the OpenAI client library, DeepSeek and Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:
# constants

MODEL_DEEPSEEK = 'deepseek-chat'
MODEL_LLAMA = 'llama3.2'  # runs locally through Ollama
OLLAMA_BASE_URL = 'http://localhost:11434/v1'

In [3]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('ANTHROPIC_AUTH_TOKEN')
# DeepSeek's OpenAI-compatible endpoint is the same host without the /anthropic suffix
base_url = os.getenv('ANTHROPIC_BASE_URL', 'https://api.deepseek.com').rstrip('/').removesuffix('/anthropic')

if not api_key:
    print("No API key was found - please check ANTHROPIC_AUTH_TOKEN in your .env file")
elif not api_key.startswith("sk-"):
    print("An API key was found, but it doesn't start sk-; please check you're using the right key")
else:
    print(f"API key found and looks good so far! Using {base_url} with model {MODEL_DEEPSEEK}")

# Both clients use the OpenAI library, but neither one talks to OpenAI
deepseek = OpenAI(api_key=api_key, base_url=base_url)
ollama = OpenAI(api_key='ollama', base_url=OLLAMA_BASE_URL)  # Ollama ignores the key, but the client requires one

system_prompt = "You are a helpful technical tutor. Explain technical questions clearly and accurately, in markdown."


def stream_answer(client, model, question):
    """Send the question to the model and show the answer in markdown as it arrives."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    stream = client.chat.completions.create(model=model, messages=messages, stream=True)
    response = ""
    handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        if chunk.choices:
            response += chunk.choices[0].delta.content or ""
            update_display(Markdown(response), display_id=handle.display_id)
    return response

API key found and looks good so far! Using https://api.deepseek.com with model deepseek-chat


In [4]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [5]:
# Get DeepSeek to answer, with streaming

deepseek_answer = stream_answer(deepseek, MODEL_DEEPSEEK, question)

# Explanation of `yield from {book.get("author") for book in books if book.get("author")}`

This is a **generator expression statement** (often used inside a generator function) that yields the **unique authors** from a collection of `books`, filtering out falsy/missing authors.

Let's break it down from the inside out.

## 1. The inner set comprehension

```python
{book.get("author") for book in books if book.get("author")}
```

This is a **set comprehension** (note the curly braces `{}`). It builds a set by iterating over `books` and:

- `for book in books` — iterates over each `book` (presumably a dict).
- `if book.get("author")` — keeps only books whose `"author"` value is truthy. This filters out:
  - Books with no `"author"` key → `get` returns `None` → falsy.
  - Books where `"author"` is `None`, `""`, `0`, etc.
- `book.get("author")` — extracts the author value (safely, without raising `KeyError` if the key is missing).

Because it's a **set**, **duplicates are automatically removed**. So if three books by "Jane Austen" exist, only one `"Jane Austen"` ends up in the set.

## 2. `yield from`

```python
yield from <iterable>
```

`yield from` delegates iteration to the given iterable. It yields every item from it, one at a time, to the caller. It's shorthand for:

```python
for item in <iterable>:
    yield item
```

Here, `yield from` yields each unique author in the set.

## 3. Ordering caveat

Because the source is a **set**, the iteration order is **not guaranteed** (sets are unordered). If you need deterministic order, use a `dict` trick or a list with deduplication:

```python
yield from dict.fromkeys(book["author"] for book in books if book.get("author"))
# or
seen = set()
for book in books:
    a = book.get("author")
    if a and a not in seen:
        seen.add(a)
        yield a
```

## 4. Where this appears

This line only works inside a **generator function** or **coroutine** (a `def` containing `yield`). Example usage:

```python
def unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

books = [
    {"title": "A", "author": "Tolkien"},
    {"title": "B", "author": "Tolkien"},
    {"title": "C", "author": "Orwell"},
    {"title": "D"},  # no author
]
print(list(unique_authors(books)))
# e.g. ['Tolkien', 'Orwell']  (order may vary)
```

## Why write it this way?

- **Concise**: one line replaces a loop + set + filter.
- **Lazy yielding**: consumers can pull authors one at a time; if used in a long pipeline, they don't need to materialize everything up front (though the inner set *is* built eagerly, so memory is used for the set).
- **Deduplication is free**: the set handles it, so downstream code doesn't need to worry about seeing the same author multiple times.
- **Safe dict access**: `.get()` avoids `KeyError` for books missing the `"author"` key.

## A subtle gotcha

The `if book.get("author")` clause **calls `.get()` twice** (once for the filter, once for the value). For plain dicts this is negligible, but for expensive or side-effecting mappings it's slightly wasteful. A cleaner version would be an explicit loop (like the `seen` example above) or using an assignment expression:

```python
yield from {(a := book.get("author")) for book in books if a}
# NOTE: walrus in a comprehension leaks/behaves subtly — verify for your Python version
```

In practice, the original line is fine for simple dicts.

In [6]:
# Get Llama 3.2 to answer, locally through Ollama

llama_answer = stream_answer(ollama, MODEL_LLAMA, question)

**Code Breakdown**
===============

The given code snippet is written in Python and utilizes the `yield from` keyword, which is part of Python 3.3 and later versions.

```python
yield from {book.get("author") for book in books if book.get("author")}
```

**What the Code Does**
---------------------

This code generates a sequence of author names from a list of books, filtering out books that do not have an "author" key.

Here's a step-by-step explanation:

1. `{book.get("author") for book in books if book.get("author")}`: This is a dictionary comprehension that filters the list of books and extracts the "author" value from each book that has an "author" key.
	* `book.get("author")`: Retrieves the value associated with the key "author" from the `book` dictionary. If the key does not exist, it returns `None`.
	* `for book in books`: Iterates over the list of books.
	* `if book.get("author")`: Filters out books that do not have an "author" key.
2. `yield from ...`: The `yield from` keyword is used to delegate to the yielded values from an iterator. In this case, it yields the values from the dictionary comprehension.

**Why the Code is Used**
-------------------------

The `yield from` keyword is useful when you need to:

* Create an iterator that delegates to another iterator.
* Avoid creating a large list or dictionary in memory.
* Iterate over a sequence of values generated by another iterator.

In this case, the code is used to generate an iterator of author names from a list of books, which can be useful in various applications, such as:

* Reading books from a database and generating an author list.
* Creating a bibliography or book metadata generator.

**Example Use Case**
-------------------

```python
books = [
    {"title": "Book 1", "author": "Author 1"},
    {"title": "Book 2", "author": "Author 2"},
    {"title": "Book 3"},
]

for author in yield from {book.get("author") for book in books if book.get("author")}:
    print(author)
```

Output:
```
Author 1
Author 2
```